# Лексическая семантика и морфологическая разметка

Лекция 3. От значений слова в WordNet к определению частей речи и оценке теггеров.

На прошлом занятии мы получали токены и леммы. Теперь добавим к ним информацию:
**какие значения есть у слова** и **какую часть речи оно представляет в предложении**.

Основные примеры — на английском: используем WordNet и размеченный корпус Brown из NLTK.
В конце вернёмся к неоднозначности в русском языке.

## Подготовка

Запускайте ячейки сверху вниз. Первая ячейка устанавливает NLTK в текущее ядро Jupyter / Colab.
Следующая скачает недостающие данные; при первом запуске нужен интернет.

In [ ]:
%pip install -q nltk

In [1]:
import nltk
from nltk.corpus import wordnet as wn, brown
from nltk.tokenize import word_tokenize

resources = {
    'punkt_tab': 'tokenizers/punkt_tab/english/',
    'wordnet': 'corpora/wordnet.zip',
    'brown': 'corpora/brown/',
    'universal_tagset': 'taggers/universal_tagset/',
    'averaged_perceptron_tagger_eng': 'taggers/averaged_perceptron_tagger_eng/',
}

for package, path in resources.items():
    try:
        nltk.data.find(path)
    except LookupError:
        nltk.download(package, raise_on_error=True)

print('NLTK:', nltk.__version__)

NLTK: 3.10.4


## WordNet

**WordNet** — лексическая база, в которой слова объединены по значениям
и связаны семантическими отношениями.

**Зачем?** Найти синонимы, перейти от частного понятия к общему, сравнить значения.

**Где применяется?** Расширение поисковых запросов, анализ значений слов,
построение словарей и семантических связей.

### Синсеты

**Синсет** — набор синонимичных лексических единиц, выражающих одно значение.
У многозначного слова несколько синсетов. Запросим значения существительного `bank`.

In [2]:
bank_synsets = wn.synsets('bank', pos=wn.NOUN)
print(bank_synsets)

[Synset('bank.n.01'), Synset('depository_financial_institution.n.01'), Synset('bank.n.03'), Synset('bank.n.04'), Synset('bank.n.05'), Synset('bank.n.06'), Synset('bank.n.07'), Synset('savings_bank.n.02'), Synset('bank.n.09'), Synset('bank.n.10')]


In [3]:
for synset in bank_synsets:
    print(synset.name(), '->', synset.definition())

bank.n.01 -> sloping land (especially the slope beside a body of water)
depository_financial_institution.n.01 -> a financial institution that accepts deposits and channels the money into lending activities
bank.n.03 -> a long ridge or pile
bank.n.04 -> an arrangement of similar objects in a row or in tiers
bank.n.05 -> a supply or stock held in reserve for future use (especially in emergencies)
bank.n.06 -> the funds held by a gambling house or the dealer in some gambling games
bank.n.07 -> a slope in the turn of a road or track; the outside is higher than the inside in order to reduce the effects of centrifugal force
savings_bank.n.02 -> a container (usually with a slot in the top) for keeping money at home
bank.n.09 -> a building in which the business of banking transacted
bank.n.10 -> a flight maneuver; aircraft tips laterally about its longitudinal axis (especially in turning)


Имя вида `bank.n.01`: слово, часть речи (`n` — noun), номер значения.
Номер идентифицирует значение в базе; он не выбирает подходящее значение для нашего текста.

In [4]:
river_bank = wn.synset('bank.n.01')
print(river_bank.definition())
print(river_bank.examples())

sloping land (especially the slope beside a body of water)
['they pulled the canoe up on the bank', 'he sat on the bank of the river and watched the currents']


`synsets()` ищет возможные значения слова, а `synset()` обращается к одному конкретному синсету.

Слова, входящие в синсет, доступны через `lemma_names()`.

In [5]:
car = wn.synset('car.n.01')
print(car.lemma_names())
print(car.definition())

['car', 'auto', 'automobile', 'machine', 'motorcar']
a motor vehicle with four wheels; usually propelled by an internal combustion engine


У слова `car` и у слова `automobile` есть общее значение.
Это не означает, что все значения этих слов взаимозаменяемы во всех контекстах.

In [6]:
print(car in wn.synsets('automobile', pos=wn.NOUN))

True


**Часть речи ещё не определяет значение.** В предложениях
`She sat on the bank.` и `She works at a bank.` слово `bank` — существительное,
но обозначает разные вещи.

Выбор значения по контексту — отдельная задача: **word sense disambiguation (WSD)**.
Сам вызов `wn.synsets()` её не решает.

### Гиперонимы и гипонимы

**Гипероним** — более общее понятие: автомобиль → транспортное средство.

**Гипоним** — более частное понятие: автомобиль → такси.

In [7]:
print('Гиперонимы:', car.hypernyms())
print('Гипонимы:', car.hyponyms()[:8])

Гиперонимы: [Synset('motor_vehicle.n.01')]
Гипонимы: [Synset('stock_car.n.01'), Synset('sports_car.n.01'), Synset('hardtop.n.01'), Synset('model_t.n.01'), Synset('cab.n.03'), Synset('minivan.n.01'), Synset('racer.n.02'), Synset('limousine.n.01')]


Путь от самого общего понятия к выбранному значению:

In [8]:
for path in car.hypernym_paths():
    print(' -> '.join(synset.name() for synset in path))

entity.n.01 -> physical_entity.n.01 -> object.n.01 -> whole.n.02 -> artifact.n.01 -> instrumentality.n.03 -> container.n.01 -> wheeled_vehicle.n.01 -> self-propelled_vehicle.n.01 -> motor_vehicle.n.01 -> car.n.01
entity.n.01 -> physical_entity.n.01 -> object.n.01 -> whole.n.02 -> artifact.n.01 -> instrumentality.n.03 -> conveyance.n.03 -> vehicle.n.01 -> wheeled_vehicle.n.01 -> self-propelled_vehicle.n.01 -> motor_vehicle.n.01 -> car.n.01


У одного синсета может быть несколько путей к корню: иерархия допускает несколько родителей.
Отношение «является разновидностью» отличается от отношения «является частью».

In [9]:
print('Части автомобиля:', car.part_meronyms())

Части автомобиля: [Synset('sunroof.n.01'), Synset('automobile_horn.n.01'), Synset('rear_window.n.01'), Synset('buffer.n.06'), Synset('fender.n.01'), Synset('glove_compartment.n.01'), Synset('floorboard.n.02'), Synset('grille.n.02'), Synset('car_window.n.01'), Synset('accelerator.n.01'), Synset('car_mirror.n.01'), Synset('first_gear.n.01'), Synset('stabilizer_bar.n.01'), Synset('car_door.n.01'), Synset('bumper.n.02'), Synset('reverse.n.02'), Synset('car_seat.n.01'), Synset('high_gear.n.01'), Synset('window.n.02'), Synset('third_gear.n.01'), Synset('tail_fin.n.02'), Synset('running_board.n.01'), Synset('air_bag.n.01'), Synset('automobile_engine.n.01'), Synset('hood.n.09'), Synset('luggage_compartment.n.01'), Synset('roof.n.02'), Synset('gasoline_engine.n.01'), Synset('auto_accessory.n.01')]


### Семантическая близость

Сравниваем **значения**, то есть синсеты.

`path_similarity()` учитывает кратчайший путь в таксономии: чем он короче, тем выше оценка.
`wup_similarity()` также учитывает глубину общего предка.

In [10]:
dog = wn.synset('dog.n.01')
cat = wn.synset('cat.n.01')
tree = wn.synset('tree.n.01')

In [11]:
for other in [cat, car, tree]:
    print(
        dog.name(), '<->', other.name(),
        '| path:', dog.path_similarity(other),
        '| wup:', dog.wup_similarity(other),
    )

dog.n.01 <-> cat.n.01 | path: 0.2 | wup: 0.8571428571428571
dog.n.01 <-> car.n.01 | path: 0.07692307692307693 | wup: 0.4
dog.n.01 <-> tree.n.01 | path: 0.125 | wup: 0.631578947368421


Это близость в словарной иерархии, а не вероятность и не универсальная оценка смысла.
Если подходящего пути нет, метод может вернуть `None`.
Выбор другого значения слова может изменить результат.

**Попробуйте:** сравните `car.n.01` с `bicycle.n.01` и `banana.n.02`.
Сначала прочитайте определения, затем объясните полученные оценки.

In [12]:
for name in ['bicycle.n.01', 'banana.n.02']:
    other = wn.synset(name)
    print(name, '->', other.definition())
    print('Близость к автомобилю:', car.path_similarity(other))

bicycle.n.01 -> a wheeled vehicle that has two wheels and is moved by foot pedals
Близость к автомобилю: 0.2
banana.n.02 -> elongated crescent-shaped yellow fruit with soft sweet flesh
Близость к автомобилю: 0.06666666666666667


## Части речи и тегсеты

**POS-тегирование** — присвоение каждому токену метки части речи в контексте.
POS = part of speech. **Тегсет** — набор допустимых меток.

**Зачем?** Часть речи помогает лемматизировать слова, находить конструкции
и готовить данные для синтаксического анализа.

In [13]:
sentences = ['I read a book.', 'They will book a room.']

for sentence in sentences:
    print(nltk.pos_tag(word_tokenize(sentence)))

[('I', 'PRP'), ('read', 'VBP'), ('a', 'DT'), ('book', 'NN'), ('.', '.')]
[('They', 'PRP'), ('will', 'MD'), ('book', 'NN'), ('a', 'DT'), ('room', 'NN'), ('.', '.')]


`book` может быть существительным или глаголом. Словаря «одно слово → одна часть речи» недостаточно.

Ожидаем для `book`: `NN` в первом предложении и `VB` во втором.
Если готовая модель во втором случае выдала `NN`, это ошибка: после `will` здесь стоит глагол.
Автоматическую разметку нужно проверять по контексту. Ниже увидим, как контекст помогает теггеру.

### Penn Treebank

Детальный тегсет для английского языка. Несколько частых обозначений:

| Тег | Значение | Пример |
| --- | --- | --- |
| NN | существительное, единственное число | book |
| NNS | существительное, множественное число | books |
| NNP | имя собственное, единственное число | London |
| VB | базовая форма глагола | read |
| VBD | прошедшее время | walked |
| VBG | форма на -ing | reading |
| JJ | прилагательное | interesting |
| RB | наречие | quickly |
| DT | определитель | the |
| PRP | личное местоимение | she |

In [14]:
tokens = word_tokenize('The students are reading interesting books.')
penn_tags = nltk.pos_tag(tokens)
print(penn_tags)

[('The', 'DT'), ('students', 'NNS'), ('are', 'VBP'), ('reading', 'VBG'), ('interesting', 'VBG'), ('books', 'NNS'), ('.', '.')]


### Universal

В NLTK `tagset='universal'` — укрупнённый набор из **12 меток**:
`NOUN`, `VERB`, `ADJ`, `ADV`, `PRON`, `DET`, `ADP`, `NUM`, `CONJ`, `PRT`, `X`, `.`.

Это не то же самое, что 17 меток UPOS в Universal Dependencies:
например, в UPOS есть отдельные `PROPN` и `AUX`, а пунктуация обозначается `PUNCT`.

In [15]:
universal_tags = nltk.pos_tag(tokens, tagset='universal')
for (word, penn), (_, universal) in zip(penn_tags, universal_tags):
    print(f'{word:14} {penn:5} -> {universal}')

The            DT    -> DET
students       NNS   -> NOUN
are            VBP   -> VERB
reading        VBG   -> VERB
interesting    VBG   -> VERB
books          NNS   -> NOUN
.              .     -> .


Укрупнение теряет информацию: `NN` и `NNS` превращаются в `NOUN`.
Качество моделей имеет смысл сравнивать на одном корпусе и в одном тегсете.

### Связь с лемматизацией

На прошлом занятии мы вручную задавали `pos` для `WordNetLemmatizer`.
Теперь получим его из POS-разметки. Обозначения Penn Treebank и WordNet различаются.

In [16]:
from nltk.stem import WordNetLemmatizer

lemmatizer = WordNetLemmatizer()

def penn_to_wordnet(tag):
    return {
        'N': wn.NOUN,
        'V': wn.VERB,
        'J': wn.ADJ,
        'R': wn.ADV,
    }.get(tag[0])

In [17]:
tagged = nltk.pos_tag(word_tokenize('The dogs were running.'))

for word, tag in tagged:
    pos = penn_to_wordnet(tag)
    lemma = lemmatizer.lemmatize(word.lower(), pos=pos) if pos else word.lower()
    print(f'{word:10} {tag:5} -> {lemma}')

The        DT    -> the
dogs       NNS   -> dog
were       VBD   -> be
running    VBG   -> run
.          .     -> .


Получился конвейер: **токенизация → POS-разметка → лемматизация**.
Ошибка в части речи может перейти на следующий шаг.

Перед POS-разметкой сохраняем регистр, пунктуацию и служебные слова:
они помогают определить контекст.

## Обучение теггеров

До этого `nltk.pos_tag()` использовал готовую английскую модель.
Теперь обучим собственные теггеры на **Brown** — корпусе с ручной POS-разметкой.

Brown имеет свой исходный тегсет. Для эксперимента сразу переведём его
в NLTK Universal и будем обучать и оценивать все модели в этих 12 метках.

In [18]:
tagged_sentences = brown.tagged_sents(categories='news', tagset='universal')
print(tagged_sentences[0])

[('The', 'DET'), ('Fulton', 'NOUN'), ('County', 'NOUN'), ('Grand', 'ADJ'), ('Jury', 'NOUN'), ('said', 'VERB'), ('Friday', 'NOUN'), ('an', 'DET'), ('investigation', 'NOUN'), ('of', 'ADP'), ("Atlanta's", 'NOUN'), ('recent', 'ADJ'), ('primary', 'NOUN'), ('election', 'NOUN'), ('produced', 'VERB'), ('``', '.'), ('no', 'DET'), ('evidence', 'NOUN'), ("''", '.'), ('that', 'ADP'), ('any', 'DET'), ('irregularities', 'NOUN'), ('took', 'VERB'), ('place', 'NOUN'), ('.', '.')]


Обучающие данные — список предложений. Каждое предложение — список пар `(токен, тег)`.
Границы предложений важны: контекст предыдущего тега не должен перескакивать между ними.

### Обучающая и тестовая выборки

На **train** модель учится, на **test** проверяем её на новых примерах.
Сначала разделим документы: предложения из одного документа попадут только в одну выборку.
Фиксируем случайное зерно, чтобы повторный запуск давал то же разделение.

Для скорости возьмём до 2000 обучающих и до 500 тестовых предложений.

In [19]:
import random

rng = random.Random(42)
fileids = sorted(brown.fileids(categories='news'))
rng.shuffle(fileids)

split = int(len(fileids) * 0.8)
train_ids = fileids[:split]
test_ids = fileids[split:]

train_sents = list(brown.tagged_sents(fileids=train_ids, tagset='universal'))
test_sents = list(brown.tagged_sents(fileids=test_ids, tagset='universal'))
rng.shuffle(train_sents)
rng.shuffle(test_sents)
train_sents = train_sents[:2000]
test_sents = test_sents[:500]

print('Документы train / test:', len(train_ids), '/', len(test_ids))
print('Предложения train / test:', len(train_sents), '/', len(test_sents))

Документы train / test: 35 / 9
Предложения train / test: 2000 / 500


Тестовые метки не передаём в обучение. Если будем подбирать параметры,
понадобится ещё и **валидационная выборка** внутри train; test оставляем для итоговой проверки.

In [20]:
sample_tokens = word_tokenize('The students read books and write reports.')
print(sample_tokens)

['The', 'students', 'read', 'books', 'and', 'write', 'reports', '.']


### Теггер по умолчанию

`DefaultTagger` присваивает всем токенам одну метку.
Возьмём самую частую метку в обучающей выборке: это простая точка отсчёта, **baseline**.

In [21]:
from collections import Counter
from nltk.tag import DefaultTagger, UnigramTagger, BigramTagger, TrigramTagger

tag_counts = Counter(tag for sent in train_sents for _, tag in sent)
most_common_tag = tag_counts.most_common(1)[0][0]
default_tagger = DefaultTagger(most_common_tag)

print('Частые метки:', tag_counts.most_common(5))
print(default_tagger.tag(sample_tokens))

Частые метки: [('NOUN', 13362), ('VERB', 6280), ('ADP', 5421), ('.', 5192), ('DET', 5010)]
[('The', 'NOUN'), ('students', 'NOUN'), ('read', 'NOUN'), ('books', 'NOUN'), ('and', 'NOUN'), ('write', 'NOUN'), ('reports', 'NOUN'), ('.', 'NOUN')]


### Униграммный теггер

`UnigramTagger` запоминает наиболее частую метку **каждого слова** в train.
Соседние слова и теги не учитывает. Для незнакомого слова без backoff вернёт `None`.

In [22]:
unigram_plain = UnigramTagger(train_sents)
print(unigram_plain.tag(sample_tokens + ['qzxword']))

[('The', 'DET'), ('students', 'NOUN'), ('read', 'VERB'), ('books', 'NOUN'), ('and', 'CONJ'), ('write', 'VERB'), ('reports', 'NOUN'), ('.', '.'), ('qzxword', None)]


**Backoff** — переход к более простому теггеру, если основной не нашёл подходящего ответа.
Для униграммного теггера используем теггер по умолчанию.

In [23]:
unigram_tagger = UnigramTagger(train_sents, backoff=default_tagger)
print(unigram_tagger.tag(sample_tokens + ['qzxword']))

[('The', 'DET'), ('students', 'NOUN'), ('read', 'VERB'), ('books', 'NOUN'), ('and', 'CONJ'), ('write', 'VERB'), ('reports', 'NOUN'), ('.', '.'), ('qzxword', 'NOUN')]


`None` исчез, но неизвестное слово всё равно может получить неверную метку.
Наличие ответа и правильность ответа — разные свойства.

### N-граммные теггеры

`BigramTagger` выбирает метку по **текущему слову и предыдущему тегу**.
`TrigramTagger` использует текущее слово и два предыдущих тега.

Здесь речь о контексте тегов, а не просто о двух или трёх соседних словах.
На этапе применения предыдущие теги уже предсказаны самой моделью.

In [24]:
bigram_tagger = BigramTagger(train_sents, backoff=unigram_tagger)
trigram_tagger = TrigramTagger(train_sents, backoff=bigram_tagger)

print('Unigram:', unigram_tagger.tag(sample_tokens))
print('Bigram: ', bigram_tagger.tag(sample_tokens))
print('Trigram:', trigram_tagger.tag(sample_tokens))

Unigram: [('The', 'DET'), ('students', 'NOUN'), ('read', 'VERB'), ('books', 'NOUN'), ('and', 'CONJ'), ('write', 'VERB'), ('reports', 'NOUN'), ('.', '.')]
Bigram:  [('The', 'DET'), ('students', 'NOUN'), ('read', 'VERB'), ('books', 'NOUN'), ('and', 'CONJ'), ('write', 'VERB'), ('reports', 'NOUN'), ('.', '.')]
Trigram: [('The', 'DET'), ('students', 'NOUN'), ('read', 'VERB'), ('books', 'NOUN'), ('and', 'CONJ'), ('write', 'VERB'), ('reports', 'NOUN'), ('.', '.')]


Цепочка: **trigram → bigram → unigram → default**.
Чем длиннее контекст, тем реже конкретное сочетание встречается в обучении.
Поэтому большее `n` не гарантирует лучшего качества.

**Маленький эксперимент.** Сделаем два употребления слова `book` одинаково частыми.
Униграммный теггер не сможет различить их по контексту.

In [25]:
toy_train = [
    [('a', 'DET'), ('book', 'NOUN')],
    [('to', 'PRT'), ('book', 'VERB')],
]
toy_unigram = UnigramTagger(toy_train, backoff=DefaultTagger('NOUN'))
toy_bigram = BigramTagger(toy_train, backoff=toy_unigram)

for words in [['a', 'book'], ['to', 'book']]:
    print(words)
    print('  unigram:', toy_unigram.tag(words))
    print('  bigram: ', toy_bigram.tag(words))

['a', 'book']
  unigram: [('a', 'DET'), ('book', 'NOUN')]
  bigram:  [('a', 'DET'), ('book', 'NOUN')]
['to', 'book']
  unigram: [('to', 'PRT'), ('book', 'NOUN')]
  bigram:  [('to', 'PRT'), ('book', 'VERB')]


Это иллюстрация механизма на двух знакомых фразах, а не оценка обобщающей способности.
Качество ниже посчитаем на отложенных документах Brown.

### Brill: обучение исправлениям

Теггер Brill начинает с базовой разметки и последовательно применяет выученные правила исправления.
Правило может учитывать соседние слова и теги.

Пример формы правила: «замени тег A на B, если предыдущий тег — C».
Конкретные A, B и C алгоритм подбирает на размеченных обучающих примерах.

In [26]:
from nltk.tag.brill import Pos, Word
from nltk.tag.brill_trainer import BrillTaggerTrainer
from nltk.tbl import Template

templates = [
    Template(Pos([-1])),
    Template(Pos([1])),
    Template(Word([-1])),
    Template(Word([1])),
]

trainer = BrillTaggerTrainer(
    initial_tagger=unigram_tagger,
    templates=templates,
    deterministic=True,
)
brill_tagger = trainer.train(train_sents[:400], max_rules=15, min_score=3)
print('Выучено правил:', len(brill_tagger.rules()))

Выучено правил: 5


Для короткого учебного запуска ограничили поиск 15 правилами и 400 предложениями из train.
`min_score=3` требует, чтобы правило уменьшало число ошибок минимум на три.
Исходный униграммный теггер обучен на всей train-выборке.

In [27]:
for rule in brill_tagger.rules()[:5]:
    print(rule)

print(brill_tagger.tag(sample_tokens))

PRT->ADP if Pos:DET@[1]
PRT->ADP if Pos:NOUN@[1]
NOUN->VERB if Pos:PRON@[-1]
PRT->ADP if Pos:NUM@[1]
ADP->PRT if Word:of@[-1]
[('The', 'DET'), ('students', 'NOUN'), ('read', 'VERB'), ('books', 'NOUN'), ('and', 'CONJ'), ('write', 'VERB'), ('reports', 'NOUN'), ('.', '.')]


Правила можно прочитать и обсудить. Но улучшение на обучающих примерах
ещё не гарантирует улучшения на новых текстах.

### Теггер на основе классификатора

Теперь представим выбор тега как классификацию: **признаки токена и контекста → метка**.
Возьмём само слово, его суффиксы, регистр, соседние слова и предыдущий тег.

`ClassifierBasedPOSTagger` по умолчанию обучает наивный байесовский классификатор.

In [28]:
from nltk.tag import ClassifierBasedPOSTagger

def pos_features(tokens, index, history):
    word = tokens[index]
    return {
        'word': word.lower(),
        'suffix2': word[-2:].lower(),
        'suffix3': word[-3:].lower(),
        'is_title': word.istitle(),
        'is_digit': word.isdigit(),
        'previous_word': tokens[index - 1].lower() if index > 0 else '<START>',
        'next_word': tokens[index + 1].lower() if index + 1 < len(tokens) else '<END>',
        'previous_tag': history[-1] if history else '<START>',
    }

In [29]:
classifier_tagger = ClassifierBasedPOSTagger(
    train=train_sents,
    feature_detector=pos_features,
)
print(classifier_tagger.tag(sample_tokens))

[('The', 'DET'), ('students', 'NOUN'), ('read', 'VERB'), ('books', 'VERB'), ('and', 'CONJ'), ('write', 'ADJ'), ('reports', 'NOUN'), ('.', '.')]


Суффиксы и регистр могут помочь с незнакомыми словами.
Следующее слово известно из входного предложения, а его правильный тег — нет:
в признаках нельзя использовать тестовую разметку.

При обучении `history` содержит эталонные предыдущие теги,
при применении — предсказанные. Поэтому ошибки могут накапливаться.

## Оценка качества тегирования

### Accuracy

Доля токенов, которым назначена правильная метка:

$$\mathrm{accuracy} = \frac{\text{число верных тегов}}{\text{число всех токенов}}$$

В знаменатель входят все токены, включая пунктуацию.
Сравним модели на одной и той же тестовой выборке.

In [30]:
models = {
    'Default': default_tagger,
    'Unigram без backoff': unigram_plain,
    'Unigram + backoff': unigram_tagger,
    'Bigram + backoff': bigram_tagger,
    'Trigram + backoff': trigram_tagger,
    'Brill': brill_tagger,
    'Naive Bayes': classifier_tagger,
}

for name, tagger in models.items():
    print(f'{name:24} {tagger.accuracy(test_sents):.2%}')

Default                  31.60%
Unigram без backoff      81.92%
Unigram + backoff        91.84%
Bigram + backoff         92.13%
Trigram + backoff        91.88%
Brill                    92.01%
Naive Bayes              89.73%


Это результаты небольшого эксперимента на новостных текстах Brown, а не рейтинг методов вообще.
Обратите внимание: Brill искал исправления только на части train, остальные обучаемые модели —
на всей train. Настройки выбраны для демонстрации, без подбора по test.

**Обсудите:** помог ли backoff? Улучшилось ли качество при переходе от bigram к trigram?
Почему более сложная модель могла проиграть?

### Известные и неизвестные слова

Общая accuracy может скрывать слабую работу на новых словах.
Посчитаем качество униграммного теггера отдельно для слов, встречавшихся и не встречавшихся в train.
Сравниваем формы с учётом регистра, как это делает наш униграммный теггер.

In [31]:
train_vocab = {word for sent in train_sents for word, _ in sent}
correct = Counter()
total = Counter()

for sent in test_sents:
    words = [word for word, _ in sent]
    predicted = unigram_tagger.tag(words)
    for (word, gold), (_, guess) in zip(sent, predicted):
        group = 'known' if word in train_vocab else 'unknown'
        total[group] += 1
        correct[group] += (gold == guess)

for group in ['known', 'unknown']:
    if total[group]:
        print(f'{group:8}: {correct[group] / total[group]:.2%}, токенов: {total[group]}')
    else:
        print(f'{group:8}: в тесте нет таких токенов')

known   : 96.03%, токенов: 9255
unknown : 67.50%, токенов: 1594


### Какие теги путает модель?

Соберём эталонные и предсказанные метки для биграммного теггера.
В строках матрицы ошибок — правильный тег, в столбцах — предсказанный.

In [32]:
gold_tags = []
predicted_tags = []

for sent in test_sents:
    words = [word for word, _ in sent]
    gold_tags.extend(tag for _, tag in sent)
    predicted_tags.extend(tag for _, tag in bigram_tagger.tag(words))

matrix = nltk.ConfusionMatrix(gold_tags, predicted_tags)
print(matrix.pretty_format(sort_by_count=True))

     |    N    V                             C         P           |
     |    O    E         A    D    A    A    O    P    R    N      |
     |    U    R         D    E    D    D    N    R    O    U      |
     |    N    B    .    P    T    J    V    J    T    N    M    X |
-----+-------------------------------------------------------------+
NOUN |<3374>  34    .    2    .   16    .    .    .    .    .    2 |
VERB |  302<1217>   .    3    .    1    .    .    .    .    .    . |
   . |    .    .<1317>   .    .    .    .    .    .    .    .    . |
 ADP |    7    1    .<1210>   1    .    5    .   71    .    .    . |
 DET |    .    .    .    5<1257>   .    .    .    .    .    .    . |
 ADJ |  186    2    .    .    . <483>  19    .    .    .    .    . |
 ADV |   28    .    .   21    3   17 <303>   .    4    .    .    . |
CONJ |    .    .    .    1    1    .    . <276>   .    .    .    . |
 PRT |    6    .    .   31    .    2    1    . <198>   .    .    . |
PRON |    1    .    .   13    4   

Диагональ — верные ответы. Вне диагонали — ошибки; посмотрим самые частые пары.

In [33]:
confusions = Counter(
    (gold, guess)
    for gold, guess in zip(gold_tags, predicted_tags)
    if gold != guess
)
for (gold, guess), count in confusions.most_common(8):
    print(f'{gold:5} -> {guess:5}: {count}')

VERB  -> NOUN : 302
ADJ   -> NOUN : 186
ADP   -> PRT  : 71
NUM   -> NOUN : 49
NOUN  -> VERB : 34
PRT   -> ADP  : 31
ADV   -> NOUN : 28
ADV   -> ADP  : 21


Чтобы объяснить ошибку, нужны сами слова и предложения.
Посмотрим первые восемь расхождений с эталоном.

In [34]:
shown = 0
for sent in test_sents:
    words = [word for word, _ in sent]
    for (word, gold), (_, guess) in zip(sent, bigram_tagger.tag(words)):
        if gold != guess:
            print(' '.join(words))
            print(f'  {word!r}: эталон={gold}, модель={guess}')
            shown += 1
            if shown >= 8:
                break
    if shown >= 8:
        break

Miss Burke , a graduate of Miss Hall's school , stayed on in Florence as a career girl .
  'on': эталон=PRT, модель=ADP
Competent teachers are well versed in the technique of leading students to pre-set conclusions without destroying the students' illusion that they are making their own decisions .
  'Competent': эталон=ADJ, модель=NOUN
Competent teachers are well versed in the technique of leading students to pre-set conclusions without destroying the students' illusion that they are making their own decisions .
  'versed': эталон=ADJ, модель=NOUN
Competent teachers are well versed in the technique of leading students to pre-set conclusions without destroying the students' illusion that they are making their own decisions .
  'to': эталон=ADP, модель=PRT
Competent teachers are well versed in the technique of leading students to pre-set conclusions without destroying the students' illusion that they are making their own decisions .
  'pre-set': эталон=ADJ, модель=NOUN
Competent teacher

Ошибки могут быть связаны с неоднозначностью, редким словом, нехваткой контекста
или несовпадением привычного нам анализа с правилами разметки корпуса.
Итоговый процент полезно дополнять ручным разбором примеров.

## Особенности русского языка

POS-разметка выбирает часть речи. Полная морфологическая разметка дополнительно
описывает падеж, число, род, время и другие признаки.

В русском языке «стали» может быть формой существительного «сталь» или глагола «стать».
Одного написания недостаточно:

- Они **стали** читать.
- Деталь сделана из **стали**.

Модель, обученная на английском Brown, не подходит для русских предложений.

На прошлом занятии `pymorphy3` возвращал возможные разборы отдельного слова.
Контекстный теггер должен выбрать разбор с учётом предложения.
Выбор первого разбора из словарного анализатора сам по себе эту задачу не решает.

**Попробуйте устно:** придумайте по два контекста для «мой», «три», «печь».
Назовите часть речи каждого употребления и слова, которые помогают её определить.

## Практика

### 1. Значения слова

Найдите синсеты `plant` как существительного. Выберите значения «растение» и «завод».
Выведите определения и гиперонимы. Объясните, почему одинаковая часть речи
не снимает семантическую неоднозначность.

In [35]:
# Ваш код здесь

### 2. Контекст и лемма

Разметьте `They saw a bird.` и `The saw is sharp.` с помощью `nltk.pos_tag()`.
Используйте `penn_to_wordnet()` и `WordNetLemmatizer`.
Сравните леммы слова `saw` и проверьте решение модели по смыслу предложений.

In [36]:
# Ваш код здесь

### 3. Анализ ошибок

Для классификатора повторите расчёт accuracy на известных и неизвестных словах.
Сравните с униграммным теггером. Выпишите три ошибки с контекстом
и предложите для каждой возможную причину.

In [37]:
# Ваш код здесь

### 4. Подумать

Модель правильно разметила 9700 токенов из 10 000.
Какова accuracy? Сколько ошибок осталось?
Достаточно ли одного этого показателя, чтобы доверять разметке всех глаголов?

## Что запомнить

- WordNet связывает значения слов; один токен может соответствовать нескольким синсетам.
- POS-теггер выбирает часть речи в контексте, а не значение слова.
- Тегсет определяет, какие различия сохраняет разметка.
- Backoff помогает получить ответ при недостатке данных; правильность ответа нужно проверить.
- Brill учит правила исправления, классификатор — связь признаков с тегом.
- Качество оцениваем на отложенных данных и разбираем ошибки на примерах.

## Материалы

- [NLTK: примеры работы с WordNet](https://www.nltk.org/howto/wordnet.html)
- [NLTK Book, глава 5: части речи и теггеры](https://www.nltk.org/book/ch05.html)
- [NLTK: последовательные теггеры и backoff](https://www.nltk.org/api/nltk.tag.sequential.html)
- [NLTK: обучение Brill](https://www.nltk.org/api/nltk.tag.brill_trainer.html)
- [NLTK: соответствия тегсетов и 12 меток Universal](https://www.nltk.org/api/nltk.tag.mapping.html)
- [Universal Dependencies: набор UPOS](https://universaldependencies.org/u/pos/)